In [1]:
import anthropic
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set API key
import os
import anthropic

from dotenv import load_dotenv
import os
import anthropic

# Load environment variables from .env
load_dotenv()

# Create Anthropic client (reads ANTHROPIC_API_KEY from .env)
client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "Say hello in one sentence."
        }
    ]
)

print("✅ Claude API connected!")
print(message.content[0].text)

✅ Claude API connected!
Hello! I'm happy to help you with whatever you need.


In [2]:
# Load the data and rebuild the model
daily_sales = pd.read_csv('../data/processed/daily_sales_CA1_FOODS.csv')
daily_sales['date'] = pd.to_datetime(daily_sales['date'])
daily_sales = daily_sales.sort_values('date').reset_index(drop=True)

FORECAST_HORIZON = 28
train = daily_sales[:-FORECAST_HORIZON].copy()
test = daily_sales[-FORECAST_HORIZON:].copy()

print("✅ Data loaded")
print(f"Test period: {test['date'].min().date()} to {test['date'].max().date()}")

✅ Data loaded
Test period: 2016-04-25 to 2016-05-22


In [13]:
# Rebuild features and tuned model (self-contained for this notebook)
from sklearn.ensemble import GradientBoostingRegressor

def create_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['quarter'] = df['date'].dt.quarter
    df['lag_7'] = df['total_sales'].shift(7)
    df['lag_14'] = df['total_sales'].shift(14)
    df['lag_28'] = df['total_sales'].shift(28)
    df['rolling_mean_7'] = df['total_sales'].shift(1).rolling(7).mean()
    df['rolling_mean_28'] = df['total_sales'].shift(1).rolling(28).mean()
    df['rolling_std_7'] = df['total_sales'].shift(1).rolling(7).std()
    return df

full_data = create_features(daily_sales.copy())

feature_cols = [
    'day_of_week', 'day_of_month', 'month', 'year',
    'week_of_year', 'is_weekend', 'quarter',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7'
]

train_fe = full_data[:-FORECAST_HORIZON].copy()
test_fe = full_data[-FORECAST_HORIZON:].copy()
train_fe = train_fe.dropna()

X_train = train_fe[feature_cols]
y_train = train_fe['total_sales']
X_test = test_fe[feature_cols]
y_test = test_fe['total_sales']

model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train)
xgb_predictions = model.predict(X_test)

print("Model retrained. Ready for anomaly detection.")

Model retrained. Ready for anomaly detection.


In [10]:
def explain_demand_anomaly(date, actual, expected, context="grocery retail"):

    pct_change = ((actual - expected) / expected) * 100
    direction = "spike" if actual > expected else "drop"

    prompt = f"""You are a supply chain demand planning expert analyzing grocery retail data.

On {date}, actual sales were {actual:.0f} units but the forecast predicted {expected:.0f} units.
This is a {abs(pct_change):.1f}% demand {direction}.

In 2-3 sentences, explain:
1. The most likely business reason for this {direction}
2. What a demand planner should do about inventory

Be specific and practical. No bullet points."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# Test it on the highest deviation day (using tuned XGBoost predictions)
test_fe = test_fe.copy()
test_fe['forecast'] = xgb_predictions
test_fe['deviation'] = test_fe['total_sales'] - test_fe['forecast']
test_fe['pct_deviation'] = abs(test_fe['deviation'] / test_fe['forecast']) * 100

# Find biggest anomaly
worst = test_fe.loc[test_fe['pct_deviation'].idxmax()]

print(f"Biggest anomaly: {worst['date'].date()}")
print(f"Actual: {worst['total_sales']:.0f} | Forecast: {worst['forecast']:.0f}")
print(f"Deviation: {worst['pct_deviation']:.1f}%")
print()
print("AI Explanation:")
print("-" * 40)
explanation = explain_demand_anomaly(
    worst['date'].date(),
    worst['total_sales'],
    worst['forecast']
)
print(explanation)

Biggest anomaly: 2016-05-15
Actual: 4717 | Forecast: 4026
Deviation: 17.2%

AI Explanation:
----------------------------------------
# Demand Spike Analysis: May 15, 2016

The 17.2% spike most likely resulted from a promotional event (holiday weekend promotion, price reduction, or advertised sale) or an external event like a holiday or severe weather that drove pantry-loading behavior, since grocery demand rarely fluctuates this sharply without a triggering event. The demand planner should immediately increase safety stock for fast-moving SKUs associated with this spike and adjust the baseline forecast upward by 10-15% for the next 4-6 weeks while investigating whether this represents a permanent demand shift or a temporary anomaly, then communicate revised replenishment quantities to the supply team to avoid stockouts without over-


In [11]:
# Define mape function (needed in this notebook too)
def mape(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    mask = actual > 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

# Calculate MAPE for the tuned XGBoost model
xgb_mape = mape(y_test, xgb_predictions)
print(f"XGBoost MAPE: {xgb_mape:.1f}%")

XGBoost MAPE: 5.7%


In [12]:
def generate_executive_summary(test_df, mape):
    """Generate an executive summary of forecast performance"""

    avg_sales = test_df['total_sales'].mean()
    max_sales = test_df['total_sales'].max()
    min_sales = test_df['total_sales'].min()
    weekend_avg = test_df[test_df['date'].dt.dayofweek >= 5]['total_sales'].mean()
    weekday_avg = test_df[test_df['date'].dt.dayofweek < 5]['total_sales'].mean()

    prompt = f"""You are a demand planning analyst presenting to a VP of Supply Chain.

28-day forecast performance for a California grocery store, FOODS category:
- Forecast accuracy: {mape:.1f}% MAPE
- Average daily sales: {avg_sales:.0f} units
- Peak day: {max_sales:.0f} units
- Lowest day: {min_sales:.0f} units
- Weekend average: {weekend_avg:.0f} units
- Weekday average: {weekday_avg:.0f} units

Write a 3-sentence executive summary that:
1. States overall forecast performance
2. Highlights the key demand pattern insight
3. Gives one specific inventory recommendation

Write for a VP. Be direct and quantitative."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

print("EXECUTIVE SUMMARY")
print("=" * 50)
summary = generate_executive_summary(test, xgb_mape)
print(summary)

EXECUTIVE SUMMARY
# Executive Summary

Our 28-day FOODS forecast for this California location delivers strong accuracy at 5.7% MAPE against an average daily demand of 3,244 units, positioning us well for inventory optimization. The data reveals a pronounced weekend spike—weekend sales average 41% higher than weekdays (4,102 vs. 2,901 units)—which is the primary driver of our 2,271-unit range between peak and trough days. We recommend increasing Thursday–Friday safety stock by 15–20% to buffer the predictable weekend surge and reduce stockouts during our highest-velocity sales window.


In [18]:
def answer_demand_question(question, data_context):
    """Answer any demand planning question using the data"""

    prompt = f"""You are a demand planning analyst with access to grocery sales data.

Data context:
{data_context}

Question from a business user: {question}

Answer in 2-3 sentences. Be specific with numbers from the data context."""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# Build data context
context = f"""
- Store: CA_1, Category: FOODS
- Test period: {test['date'].min().date()} to {test['date'].max().date()}
- Average daily sales: {test['total_sales'].mean():.0f} units
- Weekend sales are {((test[test['date'].dt.dayofweek >= 5]['total_sales'].mean() / test[test['date'].dt.dayofweek < 5]['total_sales'].mean()) - 1) * 100:.0f}% higher than weekday
- Forecast MAPE: 9.9% (seasonal naive baseline)
- Best ML model MAPE: {xgb_mape:.1f}% (gradient boosting, tuned)
"""

# Ask questions
questions = [
    "Should we increase safety stock for weekends?",
    "What's the risk of a stockout on a typical Saturday?",
    "How much would a 5% forecast improvement save in inventory costs?",
    "Should we run a promotion during low-demand weekdays?"
]

print("INTERACTIVE DEMAND Q&A")
print("=" * 50)
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {answer_demand_question(q, context)}")

INTERACTIVE DEMAND Q&A

Q: Should we increase safety stock for weekends?
A: Yes, you should increase safety stock for weekends. Weekend sales are 41% higher than weekdays, meaning weekends average approximately 4,574 units (3,244 × 1.41) versus roughly 2,324 units on weekdays. Given your best ML model achieves only 5.7% MAPE, building additional buffer stock for weekend demand spikes is a prudent inventory strategy to prevent stockouts during your highest-demand periods.

Q: What's the risk of a stockout on a typical Saturday?
A: Based on the data, a typical Saturday at CA_1 would see approximately **4,574 units** in demand (3,244 baseline × 1.41), but this comes with a ±5.7% prediction error margin from the best ML model—meaning actual demand could range roughly 4,317–4,831 units. The stockout risk depends on your current safety stock levels; if Saturday inventory falls below ~4,831 units, you face meaningful stockout exposure given the model's error band and natural demand variabilit